# Load and Consolidate ROI Stats

Walk the per-video CSVs in `working_dir/roi_stats_output/`, parse patient / region /
reconstruction from the folder hierarchy, and produce a single long-format
`consolidated.csv`.

In [1]:
from pathlib import Path

import pandas as pd

WORKING_DIR = Path.home() / "vascular-superenhancement-4d-flow" / "working_dir"
ROI_STATS_DIR = WORKING_DIR / "roi_stats_output"

NOTEBOOK_DIR = Path.home() / "vascular-superenhancement-4d-flow" / "notebooks" / "analysis" / "VSE_quantitative_analysis"
OUTPUT_CSV = NOTEBOOK_DIR / "consolidated.csv"

assert ROI_STATS_DIR.exists(), f"ROI stats directory not found: {ROI_STATS_DIR}"

## Walk directory tree and build DataFrame

Directory layout:
```
roi_stats_output/
  <patient>/
    <Region>_<reconstruction>/
      <Region>_<reconstruction>.csv   # columns: timepoint, mean, std
```

In [2]:
frames: list[pd.DataFrame] = []

for patient_dir in sorted(ROI_STATS_DIR.iterdir()):
    if not patient_dir.is_dir():
        continue
    patient = patient_dir.name

    for recording_dir in sorted(patient_dir.iterdir()):
        if not recording_dir.is_dir():
            continue
        # Parse "Ao_normal" -> region="Ao", reconstruction="normal"
        parts = recording_dir.name.rsplit("_", maxsplit=1)
        if len(parts) != 2:
            print(f"  Skipping unexpected folder name: {recording_dir.name}")
            continue
        region, reconstruction = parts

        csv_path = recording_dir / f"{recording_dir.name}.csv"
        if not csv_path.exists():
            print(f"  Missing CSV: {csv_path}")
            continue

        df = pd.read_csv(csv_path)
        df["patient"] = patient
        df["region"] = region
        df["reconstruction"] = reconstruction
        frames.append(df)

data = pd.concat(frames, ignore_index=True)

col_order = ["patient", "region", "reconstruction", "timepoint", "mean", "std"]
data = data[col_order]

print(f"Loaded {len(frames)} recordings → {len(data)} rows")
print(f"Patients: {data['patient'].nunique()}")
print(f"Regions:  {sorted(data['region'].unique())}")
print(f"Recons:   {sorted(data['reconstruction'].unique())}")
data.head()

Loaded 134 recordings → 2680 rows
Patients: 17
Regions:  ['Ao', 'Background', 'IVS', 'PA']
Recons:   ['normal', 'vse']


,patient,region,reconstruction,timepoint,mean,std
0,Balboloop,Ao,normal,1,1452.38,67.66
1,Balboloop,Ao,normal,2,1399.85,80.98
2,Balboloop,Ao,normal,3,1329.10,85.26
3,Balboloop,Ao,normal,4,1279.85,55.08
4,Balboloop,Ao,normal,5,1234.62,34.42


## Sanity checks

In [3]:
# Check for missing data and NaNs
print("Missing values per column:")
print(data.isnull().sum())
print()

# Recordings per patient (expect 8 each, except Oduskueb with 6)
recordings_per_patient = data.groupby("patient")[["region", "reconstruction"]].apply(
    lambda g: g.drop_duplicates().shape[0]
)
print("Recordings per patient:")
print(recordings_per_patient.to_string())
print()

# Verify 20 timepoints per recording
tp_counts = data.groupby(["patient", "region", "reconstruction"])["timepoint"].count()
non_twenty = tp_counts[tp_counts != 20]
if non_twenty.empty:
    print("All recordings have exactly 20 timepoints.")
else:
    print("WARNING — recordings with != 20 timepoints:")
    print(non_twenty)

Missing values per column:
patient           0
region            0
reconstruction    0
timepoint         0
mean              0
std               0
dtype: int64

Recordings per patient:
patient
Balboloop    8
Biswifo      8
Bomatog      8
Detodu       8
Diecudey     8
Diepami      8
Diequipi     8
Dublafer     8
Dujomal      8
Elagieg      8
Golotag      8
Grequafie    8
Gueshifa     8
Oduskueb     6
Quetode      8
Suquepog     8
Tiepolem     8

All recordings have exactly 20 timepoints.


## Compute SNR and save

In [4]:
data["snr"] = data["mean"] / data["std"]

NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
data.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(data)} rows to {OUTPUT_CSV}")
data.describe()

Saved 2680 rows to /home/ayeluru/vascular-superenhancement-4d-flow/notebooks/analysis/VSE_quantitative_analysis/consolidated.csv


,timepoint,mean,std,snr
count,2680.000000,2680.000000,2680.000000,2680.000000
mean,10.500000,1611.081414,121.409377,20.589114
std,5.767357,1452.361259,109.401149,338.110396
min,1.000000,47.930000,0.450000,2.226670
25%,5.750000,473.207500,47.632500,6.440372
50%,10.500000,1276.530000,85.015000,12.385379
75%,15.250000,2195.775000,161.772500,18.419082
max,20.000000,7922.660000,838.040000,17497.600000
